# swmmx: all `add` functions

This notebook is a categorized reference for every public add endpoint in `swmmx`.

Add pattern:

```python
m.add.<category>.<element_type>(id, **options)
```

- The first argument is always the new object ID.
- Implemented endpoints create records, mark the model modified, and invalidate stale results.
- Reserved endpoints are intentionally visible for autocomplete but raise `NotImplementedYetError` until implemented.
- Node-like additions require explicit `x`/`y` map coordinates.
- Subcatchments require explicit centroid `x`/`y`; an optional polygon can still define their outline.


In [ ]:
from swmmx import swmm

m = swmm(new='SI')
# Build on this scratch model when trying the examples below.


## Common add patterns

```python
m.add.node.junction('J1', x=0.0, y=0.0, invert_elevation=10.0, max_depth=3.0)
m.add.node.outfall('OUT1', x=100.0, y=0.0, invert_elevation=9.0, type='FREE')
m.add.link.conduit('C1', from_node='J1', to_node='OUT1', roughness=0.013)
m.add.hydrology.subcatchment('S1', rain_gage='RG1', outlet='J1', x=0.0, y=0.0)
```


## Hydrology

### `hydrology`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.hydrology.rain_gage(...)` | Implemented | id, format, interval, source_type | x, y, snow_catch_factor, time_series, filename, station, units | Optional x/y; defaults to maximum mapped point | created object ID (`str`) | Requires `format`, `interval`, and `source_type`; `time_series` is checked when used. |
| `m.add.hydrology.subcatchment(...)` | Implemented | id, rain_gage, outlet, x, y | area, width, slope, impervious_percent, curb_length, snow_pack, polygon, n_impervious, n_pervious, depression_storage_impervious, depression_storage_pervious, zero_depression_storage_impervious_percent, subarea_routing, percent_routed, maximum_rate, minimum_rate, decay, dry_time, maximum_volume, suction_head, hydraulic_conductivity, initial_moisture_deficit, curve_number, conductivity | Explicit centroid x/y required | created object ID (`str`) | Requires explicit centroid `x`/`y`; outlet may be a node or another subcatchment. |
| `m.add.hydrology.aquifer(...)` | Reserved; raises `NotImplementedYetError` | id | porosity, wilting_point, field_capacity, conductivity, conductivity_slope, tension_slope, upper_evaporation_fraction, lower_evaporation_depth, lower_groundwater_loss_rate, bottom_elevation, water_table_elevation, unsaturated_moisture, upper_evaporation_pattern | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydrology.snow_pack(...)` | Reserved; raises `NotImplementedYetError` | id | plowable_fraction, impervious_fraction, pervious_fraction, minimum_melt_coefficient, maximum_melt_coefficient, base_temperature, free_water_capacity_fraction, initial_snow_depth, initial_free_water, depth_at_100_percent_cover | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydrology.unit_hydrograph(...)` | Reserved; raises `NotImplementedYetError` | id, rain_gage, data | month, short_term_r, short_term_t, short_term_k, medium_term_r, medium_term_t, medium_term_k, long_term_r, long_term_t, long_term_k | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydrology.lid_control(...)` | Reserved; raises `NotImplementedYetError` | id, type | parameters, surface, pavement, soil, storage, drain, drain_mat | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |

#### `m.add.hydrology.rain_gage()`

Add a rainfall gage and its map symbol.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `format` | Required keyword | `str` enum | None | Validated according to the element definition |
| `interval` | Required keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `source_type` | Required keyword | `str` enum | None | Validated according to the element definition |
| `x` | Optional keyword | `float` | None | Validated according to the element definition |
| `y` | Optional keyword | `float` | None | Validated according to the element definition |
| `snow_catch_factor` | Optional keyword | value compatible with the named SWMM field | 1.0 | Validated according to the element definition |
| `time_series` | Optional keyword | `str` ID of existing `time_series` object | None | Must reference an existing `time_series` object |
| `filename` | Optional keyword | `str` | None | Used when `source_type='FILE'` |
| `station` | Optional keyword | `str` | None | Validated according to the element definition |
| `units` | Optional keyword | `str` | None | Validated according to the element definition |

- **INP sections:** `RAINGAGES, SYMBOLS`
- **Reference checks:** time_series -> time_series
- **Removal dependency rules:** subcatchment.rain_gage
- **Implementation status:** Implemented
- **Example:** `m.add.hydrology.rain_gage("RG1", format="INTENSITY", interval="00:05", source_type="TIMESERIES", time_series="Rain1")`

#### `m.add.hydrology.subcatchment()`

Add a subcatchment with subarea and infiltration records.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `rain_gage` | Required keyword | `str` ID of existing `rain_gage` object | None | Must reference an existing `rain_gage` object |
| `outlet` | Required keyword | `str` ID of existing `node_or_subcatchment` object | None | Must reference an existing `node_or_subcatchment` object |
| `x` | Required keyword | `float` | None | Required subcatchment centroid coordinate |
| `y` | Required keyword | `float` | None | Required subcatchment centroid coordinate |
| `area` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `width` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `slope` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `impervious_percent` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `curb_length` | Optional keyword | value compatible with the named SWMM field | 0.0 | Validated according to the element definition |
| `snow_pack` | Optional keyword | `str` ID of existing `snow_pack` object | None | Must reference an existing `snow_pack` object |
| `polygon` | Optional keyword | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Optional outline; fallback polygon is centered on x/y |
| `n_impervious` | Optional keyword | value compatible with the named SWMM field | 0.01 | Validated according to the element definition |
| `n_pervious` | Optional keyword | value compatible with the named SWMM field | 0.1 | Validated according to the element definition |
| `depression_storage_impervious` | Optional keyword | `float` | 0.05 | Validated according to the element definition |
| `depression_storage_pervious` | Optional keyword | `float` | 0.05 | Validated according to the element definition |
| `zero_depression_storage_impervious_percent` | Optional keyword | `float` | 25.0 | Validated according to the element definition |
| `subarea_routing` | Optional keyword | `float` | 'OUTLET' | Validated according to the element definition |
| `percent_routed` | Optional keyword | `float` | None | Validated according to the element definition |
| `maximum_rate` | Optional keyword | `float` | None | Validated according to the element definition |
| `minimum_rate` | Optional keyword | `float` | None | Validated according to the element definition |
| `decay` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `dry_time` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `maximum_volume` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `suction_head` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `hydraulic_conductivity` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_moisture_deficit` | Optional keyword | `float` | None | Validated according to the element definition |
| `curve_number` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `conductivity` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `SUBCATCHMENTS, SUBAREAS, INFILTRATION, POLYGONS`
- **Reference checks:** rain_gage -> rain_gage, outlet -> node_or_subcatchment, snow_pack -> snow_pack
- **Removal dependency rules:** subcatchment.outlet
- **Implementation status:** Implemented
- **Example:** `m.add.hydrology.subcatchment("S1", rain_gage="RG1", outlet="J1", x=0.0, y=0.0, area=1.0)`

#### `m.add.hydrology.aquifer()`

Reserve an aquifer definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `porosity` | Optional keyword | `float` | None | Validated according to the element definition |
| `wilting_point` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `field_capacity` | Optional keyword | `float` | None | Validated according to the element definition |
| `conductivity` | Optional keyword | `float` | None | Validated according to the element definition |
| `conductivity_slope` | Optional keyword | `float` | None | Validated according to the element definition |
| `tension_slope` | Optional keyword | `float` | None | Validated according to the element definition |
| `upper_evaporation_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `lower_evaporation_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `lower_groundwater_loss_rate` | Optional keyword | `float` | None | Validated according to the element definition |
| `bottom_elevation` | Optional keyword | `float` | None | Validated according to the element definition |
| `water_table_elevation` | Optional keyword | `float` | None | Validated according to the element definition |
| `unsaturated_moisture` | Optional keyword | `float` | None | Validated according to the element definition |
| `upper_evaporation_pattern` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `AQUIFERS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydrology.aquifer("ID", ...)`

#### `m.add.hydrology.snow_pack()`

Reserve a snow-pack definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `plowable_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `impervious_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `pervious_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `minimum_melt_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `maximum_melt_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `base_temperature` | Optional keyword | `float` | None | Validated according to the element definition |
| `free_water_capacity_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_snow_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_free_water` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `depth_at_100_percent_cover` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `SNOWPACKS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydrology.snow_pack("ID", ...)`

#### `m.add.hydrology.unit_hydrograph()`

Reserve an RDII unit hydrograph definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `rain_gage` | Required positional | `str` ID of existing `rain_gage` object | None | Must reference an existing `rain_gage` object |
| `data` | Required positional | time/value pairs, pandas Series/DataFrame, or compatible array-like data | None | Validated according to the element definition |
| `month` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `short_term_r` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `short_term_t` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `short_term_k` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `medium_term_r` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `medium_term_t` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `medium_term_k` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `long_term_r` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `long_term_t` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `long_term_k` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `HYDROGRAPHS`
- **Reference checks:** rain_gage -> rain_gage
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydrology.unit_hydrograph("ID", ...)`

#### `m.add.hydrology.lid_control()`

Reserve a multi-layer LID control definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `type` | Required positional | `str` enum | None | Validated according to the element definition |
| `parameters` | Optional positional | `dict` or structured layer data | None | Validated according to the element definition |
| `surface` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `pavement` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `soil` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `storage` | Optional keyword | `float` | None | Validated according to the element definition |
| `drain` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `drain_mat` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `LID_CONTROLS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydrology.lid_control("ID", ...)`

## Nodes

### `node`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.node.junction(...)` | Implemented | id, x, y | invert_elevation, max_depth, initial_depth, surcharge_depth, ponded_area | Explicit x/y required | created object ID (`str`) | Requires explicit map `x`/`y` coordinates. |
| `m.add.node.outfall(...)` | Implemented | id, x, y | invert_elevation, type, fixed_stage, tidal_curve, time_series, tide_gate, route_to | Explicit x/y required | created object ID (`str`) | Requires explicit map `x`/`y`; stage input depends on outfall `type`. |
| `m.add.node.divider(...)` | Reserved; raises `NotImplementedYetError` | id, x, y | invert_elevation, max_depth, initial_depth, surcharge_depth, ponded_area, type, diverted_link, cutoff_flow, diversion_curve, weir_height, weir_coefficient | Explicit x/y required | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.node.storage_unit(...)` | Reserved; raises `NotImplementedYetError` | id, x, y | invert_elevation, max_depth, initial_depth, storage_curve_type, storage_curve, area, area_coefficient, area_exponent, area_constant, evaporation_factor, seepage_loss | Explicit x/y required | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |

#### `m.add.node.junction()`

Add a hydraulic junction and a coordinate record.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `x` | Required keyword | `float` | None | Required map coordinate for node placement |
| `y` | Required keyword | `float` | None | Required map coordinate for node placement |
| `invert_elevation` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `max_depth` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `initial_depth` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `surcharge_depth` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `ponded_area` | Optional keyword | `float` | 0.0 | Validated according to the element definition |

- **INP sections:** `JUNCTIONS, COORDINATES`
- **Reference checks:** None
- **Removal dependency rules:** link.from_node, link.to_node, subcatchment.outlet
- **Implementation status:** Implemented
- **Example:** `m.add.node.junction("J1", x=0.0, y=0.0, invert_elevation=10.0, max_depth=3.0)`

#### `m.add.node.outfall()`

Add an outfall and a coordinate record.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `x` | Required keyword | `float` | None | Required map coordinate for node placement |
| `y` | Required keyword | `float` | None | Required map coordinate for node placement |
| `invert_elevation` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `type` | Optional keyword | `str` enum | 'FREE' | Validated according to the element definition |
| `fixed_stage` | Optional keyword | value compatible with the named SWMM field | None | Required only when `type='FIXED'` |
| `tidal_curve` | Optional keyword | `str` ID of existing `curve` object | None | Must reference an existing `curve` object |
| `time_series` | Optional keyword | `str` ID of existing `time_series` object | None | Must reference an existing `time_series` object |
| `tide_gate` | Optional keyword | `bool` or SWMM `YES`/`NO` string | 'NO' | Validated according to the element definition |
| `route_to` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `OUTFALLS, COORDINATES`
- **Reference checks:** tidal_curve -> curve, time_series -> time_series
- **Removal dependency rules:** link.from_node, link.to_node, subcatchment.outlet
- **Implementation status:** Implemented
- **Example:** `m.add.node.outfall("OUT1", x=100.0, y=0.0, invert_elevation=9.0, type="FREE")`

#### `m.add.node.divider()`

Reserve a flow-divider node definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `x` | Required keyword | `float` | None | Required map coordinate for node placement |
| `y` | Required keyword | `float` | None | Required map coordinate for node placement |
| `invert_elevation` | Optional keyword | `float` | None | Validated according to the element definition |
| `max_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `surcharge_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `ponded_area` | Optional keyword | `float` | None | Validated according to the element definition |
| `type` | Optional keyword | `str` enum | None | Validated according to the element definition |
| `diverted_link` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `cutoff_flow` | Optional keyword | `float` | None | Validated according to the element definition |
| `diversion_curve` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `weir_height` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `weir_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `DIVIDERS, COORDINATES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.node.divider("ID", ...)`

#### `m.add.node.storage_unit()`

Reserve a storage-unit node definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `x` | Required keyword | `float` | None | Required map coordinate for node placement |
| `y` | Required keyword | `float` | None | Required map coordinate for node placement |
| `invert_elevation` | Optional keyword | `float` | None | Validated according to the element definition |
| `max_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `storage_curve_type` | Optional keyword | `float` | None | Validated according to the element definition |
| `storage_curve` | Optional keyword | `float` | None | Validated according to the element definition |
| `area` | Optional keyword | `float` | None | Validated according to the element definition |
| `area_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `area_exponent` | Optional keyword | `float` | None | Validated according to the element definition |
| `area_constant` | Optional keyword | `float` | None | Validated according to the element definition |
| `evaporation_factor` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `seepage_loss` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `STORAGE, COORDINATES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.node.storage_unit("ID", ...)`

## Links

### `link`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.link.conduit(...)` | Implemented | id, from_node, to_node | length, roughness, shape, geometry, diameter, geometry_1, geometry_2, geometry_3, geometry_4, inlet_offset, outlet_offset, initial_flow, maximum_flow, barrels, culvert_code, entry_loss, exit_loss, average_loss, flap_gate, seepage_rate, vertices | None | created object ID (`str`) | Requires existing endpoint nodes; length can be computed from their coordinates. |
| `m.add.link.pump(...)` | Reserved; raises `NotImplementedYetError` | id, from_node, to_node, curve | initial_status, startup_depth, shutoff_depth | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.link.orifice(...)` | Reserved; raises `NotImplementedYetError` | id, from_node, to_node, type, shape, height, discharge_coefficient | width, offset, flap_gate, open_close_time | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.link.weir(...)` | Reserved; raises `NotImplementedYetError` | id, from_node, to_node, type, crest_height, discharge_coefficient | length, side_slope, flap_gate, end_contractions, end_coefficient, surcharge, road_width, road_surface | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.link.outlet(...)` | Reserved; raises `NotImplementedYetError` | id, from_node, to_node, rating_type | curve, coefficient, exponent, offset, flap_gate | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |

#### `m.add.link.conduit()`

Add a conduit with matching cross-section data.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `from_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `to_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `length` | Optional keyword | `float` | None | If omitted, computed from node coordinates when available; otherwise 1.0 |
| `roughness` | Optional keyword | `float` | 0.013 | Validated according to the element definition |
| `shape` | Optional keyword | `str` enum | 'CIRCULAR' | Validated according to the element definition |
| `geometry` | Optional keyword | numeric sequence of 1 to 4 values | None | Validated according to the element definition |
| `diameter` | Optional keyword | `float` | 1.0 | Used as circular `geometry_1` when no explicit geometry is supplied |
| `geometry_1` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `geometry_2` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `geometry_3` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `geometry_4` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `inlet_offset` | Optional keyword | value compatible with the named SWMM field | 0.0 | Validated according to the element definition |
| `outlet_offset` | Optional keyword | value compatible with the named SWMM field | 0.0 | Validated according to the element definition |
| `initial_flow` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `maximum_flow` | Optional keyword | `float` | 0.0 | Validated according to the element definition |
| `barrels` | Optional keyword | `int` | 1 | Validated according to the element definition |
| `culvert_code` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `entry_loss` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `exit_loss` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `average_loss` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `flap_gate` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |
| `seepage_rate` | Optional keyword | `float` | None | Validated according to the element definition |
| `vertices` | Optional keyword | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CONDUITS, XSECTIONS, LOSSES, VERTICES`
- **Reference checks:** from_node -> node, to_node -> node
- **Removal dependency rules:** controls
- **Implementation status:** Implemented
- **Example:** `m.add.link.conduit("C1", from_node="J1", to_node="OUT1", length=100.0, roughness=0.013, shape="CIRCULAR", diameter=1.0)`

#### `m.add.link.pump()`

Reserve a pump-link definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `from_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `to_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `curve` | Required keyword | `str` ID of existing `curve` object | None | Must reference an existing `curve` object |
| `initial_status` | Optional keyword | `str` enum | None | Validated according to the element definition |
| `startup_depth` | Optional keyword | `float` | None | Validated according to the element definition |
| `shutoff_depth` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `PUMPS`
- **Reference checks:** from_node -> node, to_node -> node, curve -> curve
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.link.pump("ID", ...)`

#### `m.add.link.orifice()`

Reserve an orifice-link definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `from_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `to_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `type` | Required keyword | `str` enum | None | Validated according to the element definition |
| `shape` | Required keyword | `str` enum | None | Validated according to the element definition |
| `height` | Required keyword | `float` | None | Validated according to the element definition |
| `discharge_coefficient` | Required keyword | `float` | None | Validated according to the element definition |
| `width` | Optional keyword | `float` | None | Validated according to the element definition |
| `offset` | Optional keyword | `float` | None | Validated according to the element definition |
| `flap_gate` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |
| `open_close_time` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `ORIFICES, XSECTIONS`
- **Reference checks:** from_node -> node, to_node -> node
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.link.orifice("ID", ...)`

#### `m.add.link.weir()`

Reserve a weir-link definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `from_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `to_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `type` | Required keyword | `str` enum | None | Validated according to the element definition |
| `crest_height` | Required keyword | `float` | None | Validated according to the element definition |
| `discharge_coefficient` | Required keyword | `float` | None | Validated according to the element definition |
| `length` | Optional keyword | `float` | None | Validated according to the element definition |
| `side_slope` | Optional keyword | `float` | None | Validated according to the element definition |
| `flap_gate` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |
| `end_contractions` | Optional keyword | `int` | None | Validated according to the element definition |
| `end_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `surcharge` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |
| `road_width` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `road_surface` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `WEIRS`
- **Reference checks:** from_node -> node, to_node -> node
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.link.weir("ID", ...)`

#### `m.add.link.outlet()`

Reserve an outlet-link definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `from_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `to_node` | Required keyword | `str` ID of existing `node` object | None | Must reference an existing `node` object |
| `rating_type` | Required keyword | `str` enum | None | Validated according to the element definition |
| `curve` | Optional keyword | `str` ID of existing `curve` object | None | Must reference an existing `curve` object |
| `coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `exponent` | Optional keyword | `float` | None | Validated according to the element definition |
| `offset` | Optional keyword | `float` | None | Validated according to the element definition |
| `flap_gate` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |

- **INP sections:** `OUTLETS`
- **Reference checks:** from_node -> node, to_node -> node, curve -> curve
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.link.outlet("ID", ...)`

## Hydraulics

### `hydraulic`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.hydraulic.street(...)` | Reserved; raises `NotImplementedYetError` | id | crown_width, curb_height, cross_slope, roughness, depression_storage, gutter_width, gutter_slope | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydraulic.inlet(...)` | Reserved; raises `NotImplementedYetError` | id | type, grate_length, grate_width, grate_type, curb_length, curb_height, slotted_length, slotted_width | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydraulic.transect(...)` | Reserved; raises `NotImplementedYetError` | id | roughness_left, roughness_right, roughness_channel, left_bank, right_bank, stations, elevations, modifiers | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.hydraulic.control(...)` | Reserved; raises `NotImplementedYetError` | id | text, conditions, actions, priority | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |

#### `m.add.hydraulic.street()`

Reserve a street definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `crown_width` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `curb_height` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `cross_slope` | Optional keyword | `float` | None | Validated according to the element definition |
| `roughness` | Optional keyword | `float` | None | Validated according to the element definition |
| `depression_storage` | Optional keyword | `float` | None | Validated according to the element definition |
| `gutter_width` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `gutter_slope` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `STREETS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydraulic.street("ID", ...)`

#### `m.add.hydraulic.inlet()`

Reserve an inlet definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `type` | Optional keyword | `str` enum | None | Validated according to the element definition |
| `grate_length` | Optional keyword | `float` | None | Validated according to the element definition |
| `grate_width` | Optional keyword | `float` | None | Validated according to the element definition |
| `grate_type` | Optional keyword | `float` | None | Validated according to the element definition |
| `curb_length` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `curb_height` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `slotted_length` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `slotted_width` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `INLETS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydraulic.inlet("ID", ...)`

#### `m.add.hydraulic.transect()`

Reserve a transect definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `roughness_left` | Optional keyword | `float` | None | Validated according to the element definition |
| `roughness_right` | Optional keyword | `float` | None | Validated according to the element definition |
| `roughness_channel` | Optional keyword | `float` | None | Validated according to the element definition |
| `left_bank` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `right_bank` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `stations` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `elevations` | Optional keyword | `float` | None | Validated according to the element definition |
| `modifiers` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `TRANSECTS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydraulic.transect("ID", ...)`

#### `m.add.hydraulic.control()`

Reserve a control-rule definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `text` | Optional positional | `str` | None | Validated according to the element definition |
| `conditions` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `actions` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `priority` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `CONTROLS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.hydraulic.control("ID", ...)`

## Water quality

### `quality`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.quality.pollutant(...)` | Reserved; raises `NotImplementedYetError` | id | units, rain_concentration, groundwater_concentration, rdii_concentration, decay_coefficient, snow_only, co_pollutant, co_pollutant_fraction, dry_weather_flow_concentration, initial_concentration | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.quality.land_use(...)` | Reserved; raises `NotImplementedYetError` | id | sweeping_interval, sweeping_availability, last_swept | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |

#### `m.add.quality.pollutant()`

Reserve a pollutant definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `units` | Optional keyword | `str` | None | Validated according to the element definition |
| `rain_concentration` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `groundwater_concentration` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `rdii_concentration` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `decay_coefficient` | Optional keyword | `float` | None | Validated according to the element definition |
| `snow_only` | Optional keyword | `bool` or SWMM `YES`/`NO` string | None | Validated according to the element definition |
| `co_pollutant` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `co_pollutant_fraction` | Optional keyword | `float` | None | Validated according to the element definition |
| `dry_weather_flow_concentration` | Optional keyword | `float` | None | Validated according to the element definition |
| `initial_concentration` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |

- **INP sections:** `POLLUTANTS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.quality.pollutant("ID", ...)`

#### `m.add.quality.land_use()`

Reserve a land-use definition.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `sweeping_interval` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `sweeping_availability` | Optional keyword | `float` | None | Validated according to the element definition |
| `last_swept` | Optional keyword | `float` | None | Validated according to the element definition |

- **INP sections:** `LANDUSES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Reserved; raises `NotImplementedYetError`
- **Example:** `m.add.quality.land_use("ID", ...)`

## Curves

### `curve`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.curve.control(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.diversion(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.pump(...)` | Implemented | id, points | curve_type | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.rating(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.shape(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.storage(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.tidal(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.weir(...)` | Implemented | id, points | None | None | created object ID (`str`) | Use the listed required inputs; unsupported reserved endpoints raise a clear error. |
| `m.add.curve.generic(...)` | Implemented | id, type, points | None | None | created object ID (`str`) | Requires explicit curve `type` plus x/y `points`. |

#### `m.add.curve.control()`

Add a control curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.control("ID", ...)`

#### `m.add.curve.diversion()`

Add a diversion curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.diversion("ID", ...)`

#### `m.add.curve.pump()`

Add a pump curve from x/y points.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |
| `curve_type` | Optional keyword | `str` enum | 'PUMP1' | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.pump("PumpCurve1", [(0.0, 0.0), (1.0, 2.0)])`

#### `m.add.curve.rating()`

Add a rating curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.rating("ID", ...)`

#### `m.add.curve.shape()`

Add a shape curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.shape("ID", ...)`

#### `m.add.curve.storage()`

Add a storage curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.storage("ID", ...)`

#### `m.add.curve.tidal()`

Add a tidal curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.tidal("ID", ...)`

#### `m.add.curve.weir()`

Add a weir curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.weir("ID", ...)`

#### `m.add.curve.generic()`

Add a generic explicitly-typed curve.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `type` | Required positional | `str` enum | None | Explicit SWMM curve type |
| `points` | Required positional | `list[tuple[float, float]]`, pandas DataFrame, or NumPy `(n, 2)` array | None | Validated according to the element definition |

- **INP sections:** `CURVES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.curve.generic("ID", ...)`

## Time data

### `time`

| Add function | Status | Required inputs | Optional inputs | Coordinate rule | Output | Conditions |
| --- | --- | --- | --- | --- | --- | --- |
| `m.add.time.time_series(...)` | Implemented | id | data, datetime, values, filename, description | None | created object ID (`str`) | Accepts inline time/value data or filename-based series metadata. |
| `m.add.time.time_pattern(...)` | Implemented | id, type, multipliers | None | None | created object ID (`str`) | Pattern `type` must be MONTHLY, DAILY, HOURLY, or WEEKEND. |

#### `m.add.time.time_series()`

Add a time series from timestamp/value data or a source file.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `data` | Optional positional | time/value pairs, pandas Series/DataFrame, or compatible array-like data | None | Validated according to the element definition |
| `datetime` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `values` | Optional keyword | value compatible with the named SWMM field | None | Validated according to the element definition |
| `filename` | Optional keyword | `str` | None | Validated according to the element definition |
| `description` | Optional keyword | `str` | None | Validated according to the element definition |

- **INP sections:** `TIMESERIES`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.time.time_series("Rain1", data=[("2026-01-01 00:00", 0.0), ("2026-01-01 00:05", 5.0)])`

#### `m.add.time.time_pattern()`

Add a monthly, daily, hourly, or weekend time pattern.

| Input | Role | Expected type | Default | Conditions / validation |
| --- | --- | --- | --- | --- |
| `id` | Required first positional | `str` | None | Validated according to the element definition |
| `type` | Required positional | `str` enum | None | Validated according to the element definition |
| `multipliers` | Required positional | 1D numeric sequence | None | Validated according to the element definition |

- **INP sections:** `PATTERNS`
- **Reference checks:** None
- **Removal dependency rules:** None
- **Implementation status:** Implemented
- **Example:** `m.add.time.time_pattern("Pat1", "DAILY", [1.0] * 24)`